In [0]:
-- 1. Read, Type, Trim, and generate Surrogate Key (Streaming)
CREATE TEMPORARY STREAMING TABLE stg_products_typed AS
SELECT 
  -- Generate a unique Surrogate Key for this specific version of the product record
  md5(concat_ws('|', TRIM(product_id), CAST(_ingestion_timestamp AS STRING))) AS product_sk,
  
  _id,
  CAST(TRIM(product_id) AS INT) AS product_id,
  TRIM(product_name) AS product_name,
  TRIM(product_brand) AS product_brand,
  TRIM(product_category) AS product_category,
  TRIM(product_sku) AS product_sku,
  
  -- Invalid strings like 'FREE' or '10 lbs' will cleanly cast to NULL
  CAST(TRIM(product_cost) AS DOUBLE) AS product_cost,
  CAST(TRIM(product_retail_price) AS DOUBLE) AS product_retail_price,
  CAST(TRIM(product_weight) AS DOUBLE) AS product_weight,
  
  -- Robust BOOLEAN casting to handle messy source flag strings
  CASE 
    WHEN UPPER(TRIM(low_fat)) IN ('1', 'TRUE', 'YES', 'Y') THEN TRUE
    WHEN UPPER(TRIM(low_fat)) IN ('0', 'FALSE', 'NO', 'N') THEN FALSE
    ELSE NULL 
  END AS low_fat,
  
  CASE 
    WHEN UPPER(TRIM(recyclable)) IN ('1', 'TRUE', 'YES', 'Y') THEN TRUE
    WHEN UPPER(TRIM(recyclable)) IN ('0', 'FALSE', 'NO', 'N') THEN FALSE
    ELSE NULL 
  END AS recyclable,
  
  _ingestion_timestamp,
  _source_system
FROM STREAM(mavenmarket.bronze.products);

-- 2. Run Generic KPI Checks (Completeness, Validity, Consistency)
CREATE TEMPORARY STREAMING TABLE stg_products_checked AS
SELECT 
  *,
  -- HARD CHECKS (Route to Quarantine)
  -- Note: The CAST to INT above will turn invalid IDs like 'ERR_99' to NULL, triggering this check
  CASE WHEN product_id IS NULL THEN 'Completeness: product_id is null; ' ELSE '' END AS err_product_id,
  CASE WHEN product_retail_price < 0 THEN 'Validity: product_retail_price is negative; ' ELSE '' END AS err_price_neg,
  
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN product_name IS NULL THEN 'Completeness: product_name is null; ' ELSE '' END AS warn_name,
  CASE WHEN product_cost > product_retail_price THEN 'Consistency: product_cost exceeds product_retail_price; ' ELSE '' END AS warn_margin
FROM STREAM(LIVE.stg_products_typed);

-- 3. Consolidate Flags
CREATE TEMPORARY STREAMING TABLE stg_products_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_product_id, err_price_neg)) AS hard_failures,
  TRIM(CONCAT(warn_name, warn_margin)) AS dq_flags
FROM STREAM(LIVE.stg_products_checked);

-- 4. Quarantine Table Setup (Records failing hard checks)
CREATE STREAMING TABLE quarantine_products
COMMENT "Quarantined products failing hard checks"
AS SELECT 
  * EXCEPT(err_product_id, err_price_neg, warn_name, warn_margin, dq_flags, hard_failures),
  hard_failures AS quarantine_reasons
FROM STREAM(LIVE.stg_products_flagged)
WHERE hard_failures != '';

-- 5. Valid Products Stream Setup (Records passing hard checks)
CREATE TEMPORARY STREAMING TABLE stg_products_valid AS
SELECT 
  * EXCEPT(err_product_id, err_price_neg, warn_name, warn_margin, hard_failures)
FROM STREAM(LIVE.stg_products_flagged)
WHERE hard_failures = '';

-- 6. Target Silver Table Declaration & SCD Type 2 Implementation
CREATE STREAMING TABLE silver_products
COMMENT "Cleaned product master data with warn-level data quality flags. Managed as an SCD Type 2 Dimension.";

APPLY CHANGES INTO LIVE.silver_products
FROM STREAM(LIVE.stg_products_valid)
KEYS (product_id)
SEQUENCE BY _ingestion_timestamp
STORED AS SCD TYPE 2;